In [ ]:
!git clone https://github.com/Arijit1080/Licence-Plate-Detection-and-Recognition-using-YOLO-V8-EasyOCR.git

In [ ]:
cd /content/Licence-Plate-Detection-and-Recognition-using-YOLO-V8-EasyOCR

In [ ]:
pip install -r requirements.txt

In [ ]:
!python predictWithOCR.py model='/content/Licence-Plate-Detection-and-Recognition-using-YOLO-V8-EasyOCR/best.pt' source='/content/Licence-Plate-Detection-and-Recognition-using-YOLO-V8-EasyOCR/image16.jpg'

In [ ]:
import re
import cv2
import easyocr
import numpy as np
from google.cloud import vision
import io
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/numberplaterecognition-458117-73ec1e7e82b9.json"


def extract_features(image_path, template_path="/content/Template.png"):
    """Extracts features from the image using the template."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    features = {}
    for template_path in template_paths:
        template = cv2.imread(template_path, cv2.IMREAD_GRAYSCALE)
        if img is None or template is None:
            print(f"Warning: Could not load image or template ({template_path}). Skipping feature extraction.")
            continue  # Skip to the next template

        # Feature extraction logic (example - adapt to your needs):
        img_h, img_w = img.shape
        template_h, template_w = template.shape

        features[f'{template_path}_font_size_ratio'] = img_h / template_h
        features[f'{template_path}_aspect_ratio'] = img_w / img_h

        img_edges = cv2.Canny(img, 100, 200)
        template_edges = cv2.Canny(template, 100, 200)
        features[f'{template_path}_edge_ratio'] = np.sum(img_edges) / np.sum(template_edges)

    return features

def recognize_number_plate(image_path):
    """Recognizes the text in a number plate image."""

    reader = easyocr.Reader(['en', 'hi'], gpu=True)

    # Use Google Cloud Vision API for better accuracy
    client = vision.ImageAnnotatorClient()
    with io.open(image_path, 'rb') as image_file:
      content = image_file.read()
    image = vision.Image(content=content)
    response = client.text_detection(image=image)

    # Extract plate text from Google Vision response
    plate_text = ""
    for text in response.text_annotations:
      plate_text = text.description
      break  # Assuming the first annotation is the entire plate text

    return plate_text


def validate_number_plate(image_path, plate_text, template_paths):
    """Validates the number plate using extracted features from multiple templates."""
    plate_text = plate_text.replace('\n', '').replace(' ', '').upper()
    features = extract_features(image_path, template_paths)

    # Validation logic using features from multiple templates (example):
    is_legit = True  # Assume legit initially
    for template_path in template_paths:
        font_size_ratio_key = f'{template_path}_font_size_ratio'
        aspect_ratio_key = f'{template_path}_aspect_ratio'
        edge_ratio_key = f'{template_path}_edge_ratio'

        # Check if features for this template were extracted successfully
        if font_size_ratio_key not in features or aspect_ratio_key not in features or edge_ratio_key not in features:
            continue  # Skip to the next template if features are missing

        font_size_ratio = features[font_size_ratio_key]
        aspect_ratio = features[aspect_ratio_key]
        edge_ratio = features[edge_ratio_key]

        # Apply your validation logic using features and thresholds
        # Example thresholds - adjust based on your templates
        if not (0.8 < font_size_ratio < 1.2):
            is_legit = False
            break  # Stop checking if not legit for this template
        if not (2.5 < aspect_ratio < 4.5):
            is_legit = False
            break
        if not (0.5 < edge_ratio < 1.5):
            is_legit = False
            break

    return "Legit" if is_legit else "Damaged/Tampered"  # Return Legit or Damaged/Tampered


# usage
template_paths = ["/content/Template.png", "/content/Template2.png"]
image_path = "/content/image.png"
plate_text = recognize_number_plate(image_path)
status = validate_number_plate(image_path, plate_text, template_paths)

print("Number Plate Text:", plate_text)
print("Status:", status)

In [ ]:
!pip install easyocr google-cloud-vision opencv-python-headless

In [ ]:
import re
import cv2
import easyocr
import numpy as np
from google.cloud import vision
import io
import os

In [ ]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/numberplaterecognition-458117-73ec1e7e82b9.json"

In [ ]:
import re
import cv2
import easyocr
import numpy as np
from google.cloud import vision
import io
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/numberplaterecognition-458117-73ec1e7e82b9.json"

# Initialize EasyOCR reader outside the function to reuse it
reader = easyocr.Reader(['en', 'hi'], gpu=True)

# Initialize Google Cloud Vision client outside the function to reuse it
client = vision.ImageAnnotatorClient()

def recognize_number_plate(image_path):
    """Recognizes the text in a number plate image."""

    # --- Grayscale Conversion and Storage ---
    img = cv2.imread(image_path)  # Load the image using OpenCV
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    cv2.imwrite(image_path, gray)  # Overwrite original with grayscale

    # Use Google Cloud Vision API for better accuracy (on grayscale image)
    with io.open(image_path, 'rb') as image_file:
        content = image_file.read()
    image = vision.Image(content=content)
    response = client.text_detection(image=image)

    # Extract plate text from Google Vision response
    plate_text = ""
    for text in response.text_annotations:
        plate_text = text.description
        break  # Assuming the first annotation is the entire plate text

    # --- Pass Grayscale Image to EasyOCR --- (Only if Google Vision fails)
    if not plate_text:  # Check if Google Vision found any text
        result = reader.readtext(gray)
        for detection in result:
            plate_text += detection[1]

    return plate_text

def validate_number_plate(plate_text):
    """Validates the number plate using character count."""
    plate_text = plate_text.replace('\n', '').replace(' ', '').upper()
    # print(len(plate_text))

    # Missing Characters Check:
    # Indian HSRP number plates have 13 characters
    # considering older number plates that consists of 9 to 10 characters
    # 25 characters including the laser encrypted text
    if len(plate_text) != 13 and len(plate_text) != 9 and len(plate_text) != 10 and len(plate_text) != 25 and len(plate_text) != 11:
        return "Tampered"  # Consider it tampered if characters are missing
    else:
        return "Legit"


# Usage
image_path = "/content/image22.jpg"
plate_text = recognize_number_plate(image_path)
status = validate_number_plate(plate_text)
print("Number Plate Text:", plate_text)
print("Status:", status)

Number Plate Text: AP27
058
Status: Tampered
